# 11 · Two more independent lung cohorts (plan 7j, part A)

**Project:** Acquisition shortcuts in public cancer imaging (Hamza El Massaoudy)
**Notebook written:** 3 Nov 2026 · **Needs:** `results/01_inventory/` to `results/09_transfer_posthoc/` · **Plan:**
section **7j**, part A (written on 3 Nov 2026 before any image, embedding or model output of these cohorts existed).

**What this notebook does, in plain words**

Notebook 08 trained models on the flagged TCGA lung benchmark (TCGA-LUAD vs TCGA-LUSC) and tested them on two other
lung datasets: they **failed on an independent cohort** (NSCLC Radiogenomics) but **passed on CPTAC lung**, which was
assembled the same confounded way, even from the background alone. One independent cohort is thin evidence, so this
notebook adds two more **independent, single-source cohorts** that ask the same question:
- **NSCLC-Radiomics** (Lung1): adenocarcinoma 51 vs squamous cell carcinoma 152 (labels: Lung1 clinical file, pinned);
- **Lung-PET-CT-Dx**: adenocarcinoma (A) 251 vs squamous cell carcinoma (G) 61 (labels: the patient-ID code).

For each of the four lung targets, both encoders and the full image, the background-only image and (secondary) the
silhouette, it reports the external AUC, the transfer gap and the target's own AUC, with notebook 08's rules. The two
old targets are recomputed and must reproduce notebooks 08 and 09. Plan 7j's reading rule is then applied, and the
slice-thickness prediction of 7j A.6 is checked (descriptive).

**Rehearsal first.** With the default `RUN_MODE = "both"`, everything first runs on **shuffled labels**, checks itself
against notebooks 05-09, and only then runs on the real labels.

**Amendment, 4 Nov 2026 (label-blind, before any real-label result):** for 17 Lung-PET-CT-Dx patients the series rule
picks a Siemens `<ALPHA Range>` series, which holds Secondary Capture renderings with no slice geometry, not CT slices.
The first two Colab runs therefore stopped at the coverage gate (295 of 312). Such a series is now set aside and the same
rule picks again from the patient's other series; every replacement is recorded (`cache/series_set_aside.csv` and the
run manifest). Nothing else changed.

**How to run:** in Colab choose *Runtime → Change runtime type → L4 GPU (or A100), High-RAM*, then *Runtime → Run all*.
Sign in with **the Google account that holds the project Drive folder** when Drive asks. Expect about 3–4.5 hours: about 2.5–3.5 hours to download and
prepare about 515 new scans (about 50 GB, on Colab's own disk; each scan is deleted once embedded), then about
15–30 minutes per pass. **If Colab disconnects, run all again:** finished scans and finished pairs are loaded, not
redone. Outputs go to `results/11_external_lung/`. The last cell stops the runtime.

In [1]:
# 1. Setup: Drive, folders, GPU
import os, re, sys, json, time, copy, shutil, zipfile, hashlib, datetime, platform, warnings, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
    WORK = Path('/content/asa11_work')          # Colab's own disk: DICOM and NIfTI files, never Drive
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    WORK = Path(os.environ['ASA_WORK_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')
assert RUN_MODE in ('both', 'rehearsal', 'real'), RUN_MODE

IN01 = PROJECT / 'results' / '01_inventory'
IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
IN04 = PROJECT / 'results' / '04_posthoc'
IN05 = PROJECT / 'results' / '05_image_check'
IN06 = PROJECT / 'results' / '06_image_posthoc'
IN07 = PROJECT / 'results' / '07_screen_validation'
IN08 = PROJECT / 'results' / '08_transfer'
IN09 = PROJECT / 'results' / '09_transfer_posthoc'
CACHE05, CACHE06, CACHE07, CACHE08 = IN05 / 'cache', IN06 / 'cache', IN07 / 'cache', IN08 / 'cache'   # read only
OUT_ROOT = PROJECT / 'results' / '11_external_lung'
CACHE = OUT_ROOT / 'cache'                       # label-free: TCIA replies, headers, embeddings and QC pictures
LABEL_DIR = OUT_ROOT / 'labels'                  # the pinned Lung1 label file
for d in (IN01, IN02, IN03, IN04):
    assert (d / 'run_manifest.json').exists(), f'{d} is missing: run the earlier notebooks first.'
for d in (IN05, IN06, IN07, IN08, IN09):
    for m in ('rehearsal', 'real'):
        assert (d / m / 'run_manifest.json').exists(), f'{d / m} is missing: run notebooks 05 to 09 first.'
for d in (CACHE / 'api', CACHE / 'headers', CACHE / 'embeddings', CACHE / 'qc', LABEL_DIR, WORK / 'nifti', WORK / 'dicom'):
    d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
free_gb = shutil.disk_usage(WORK).free / 1e9
print('Project folder:', PROJECT)
print('Outputs to    :', OUT_ROOT)
print('Work disk     :', WORK, f'({free_gb:.0f} GB free)')
print('Run mode      :', RUN_MODE)
print('Run started   :', RUN_STARTED, '(UTC)')
if free_gb < 80:
    print('WARNING: less than 80 GB free on the work disk. The prepared scans need about 30 GB there at the peak. '
          'Use an L4 or A100 runtime (they have more disk).')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/11_external_lung
Work disk     : /content/asa11_work (253 GB free)
Run mode      : both
Run started   : 2026-11-03T16:31:30+00:00 (UTC)


In [2]:
# 2. Packages (exact versions; installed into Colab only) (notebook 08, unchanged)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai==1.6.1', 'SimpleITK==2.5.6',
                    'pydicom==3.0.2', 'nibabel==5.4.2', 'merlin-vlm==0.0.7'], check=True)

import numpy as np
import pandas as pd
import requests
import scipy
from scipy import ndimage, sparse
from scipy.stats import rankdata, spearmanr
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed
import pydicom
import SimpleITK as sitk
import nibabel
import torch
import torchvision
import monai
from monai.transforms import (Compose, LoadImage, EnsureType, Orientation, Spacing, ScaleIntensityRange, SpatialCrop)
from monai.transforms.utils import generate_spatial_bounding_box
from monai.inferers import SlidingWindowSplitter
from monai.networks.nets.segresnet_ds import SegResEncoder
import huggingface_hub
from huggingface_hub import hf_hub_download
import safetensors.torch
import merlin
from merlin.models import i3res
from merlin.data.monai_transforms import ImageTransforms as MERLIN_TRANSFORMS
import importlib.metadata

warnings.filterwarnings('ignore', message='.*pixdim.*')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1
N_CPU = os.cpu_count()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, default=_jsonable))
    tmp.replace(path)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__,
            'torch': torch.__version__, 'torchvision': torchvision.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version.VersionString(), 'pydicom': pydicom.__version__,
            'nibabel': nibabel.__version__, 'huggingface_hub': huggingface_hub.__version__,
            'merlin-vlm': importlib.metadata.version('merlin-vlm'), 'cpus': N_CPU,
            'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'}
print(VERSIONS)
if IN_COLAB and DEVICE != 'cuda':
    print('WARNING: no GPU. Choose Runtime → Change runtime type → L4 GPU, then run all again.')

{'python': '3.13.16', 'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'joblib': '1.6.0', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'monai': '1.6.1', 'SimpleITK': '2.5.6', 'pydicom': '3.0.2', 'nibabel': '5.4.2', 'huggingface_hub': '1.33.0', 'merlin-vlm': '0.0.7', 'cpus': 12, 'gpu': 'NVIDIA A100-SXM4-80GB'}


/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


## Settings (plan sections 7d, 7f, 7g, 7h and 7j)
Every model, image and encoder setting is notebook 05's (checked against its run record below); the silhouette is
notebook 06's; the transfer rule and statistics are notebook 08's; the field comparison is notebook 09's. New here:
the two cohorts of plan 7j, their pinned labels and expected counts, and the 7j reading rule.

In [3]:
# 3. Settings
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part',
               'hdr_kernel', 'hdr_filter_type', 'hdr_patient_position'}
LOG_NUMERIC = {'image_count', 'body_area_cm2'}
TEXT = {'protocol_text', 'care_text'}

TCIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
PREP_JOBS = max(1, min(8, N_CPU - 1))
EMB_WORKERS = max(1, min(6, N_CPU - 2)) if IN_COLAB else 0
BODY_HU, OPEN_MM, DILATE_MM, AIR_HU = -500, 3.0, 5.0, -1000
N_QC = 2
CTFM_REPO, CTFM_REVISION = 'project-lighter/ct_fm_feature_extractor', '527b0d1c262b433396c85ff2de84c427b26945a1'
CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP, CTFM_BATCH = (3.0, 1.0, 1.0), (24, 128, 128), 0.625, 32
MERLIN_REPO, MERLIN_REVISION = 'stanfordmimi/Merlin', '66dd07585de79e4d8d70d67806da88f3ccaa5a0c'
MERLIN_CKPT = 'i3_resnet_clinical_longformer_best_clip_04-02-2024_23-21-36_epoch_99.pt'
ENCODERS = ('ctfm', 'merlin')
HDR_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_pixel_spacing', 'hdr_recon_diameter', 'hdr_kvp',
               'hdr_tube_current', 'hdr_exposure', 'hdr_ctdivol', 'hdr_pitch']
HDR_CATEGORICAL = ['hdr_kernel', 'hdr_filter_type', 'hdr_patient_position']
HDR_COLS = HDR_NUMERIC + HDR_CATEGORICAL
GATE_NULL_RANGE = (0.35, 0.55)
GATE_MIN_COVERAGE = 0.95
MAX_SERIES_SET_ASIDE = 3   # amendment of 4 Nov 2026: series without CT slices set aside per patient, at most
REPRO_TOL = 1e-3            # as notebooks 06 to 09
SIL_HU = 0                  # notebook 06's silhouette

PREP_KEY05 = hashlib.md5(json.dumps({'BODY_HU': BODY_HU, 'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU,
                                     'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY05 = hashlib.md5(json.dumps({'prep': PREP_KEY05, 'ctfm': [CTFM_REVISION, CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP],
                                    'merlin': [MERLIN_REVISION, MERLIN_CKPT], 'v': 1},
                                   sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY = hashlib.md5(json.dumps({'nb05': EMB_KEY05, 'SIL_HU': SIL_HU, 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
PREP_KEY = hashlib.md5(json.dumps({'nb05': PREP_KEY05, 'SIL_HU': SIL_HU, 'nb08': 1},
                                  sort_keys=True).encode()).hexdigest()[:12]   # notebook 08's, unchanged
VARIANTS = ('original', 'background', 'silhouette')       # prepared and embedded, as in notebooks 07 and 08
INPUTS_B = ('original', 'background')                     # notebook 08's transfer analysis
INPUT_NAME = {'original': 'image', 'background': 'background', 'silhouette': 'silhouette'}

# The lung pairs: the source is notebook 05's flagged task; two old targets (notebook 08) and two new ones (plan 7j)
SOURCE = 'lung histology (LUAD vs LUSC)'
SOURCES = [SOURCE]
NSCLC_TASK = 'NSCLC Radiogenomics | histology'
OLD_TARGETS = ('NSCLC Radiogenomics', 'CPTAC lung')
NEW_TARGETS = ('NSCLC-Radiomics', 'Lung-PET-CT-Dx')
PAIR_OF = {'NSCLC Radiogenomics': 'lung: TCGA -> NSCLC Radiogenomics', 'CPTAC lung': 'lung: TCGA -> CPTAC',
           'NSCLC-Radiomics': 'lung: TCGA -> NSCLC-Radiomics', 'Lung-PET-CT-Dx': 'lung: TCGA -> Lung-PET-CT-Dx'}
PAIRS = [{'pair': PAIR_OF[t], 'source': SOURCE, 'target': t} for t in OLD_TARGETS + NEW_TARGETS]
ROLE = {'NSCLC Radiogenomics': 'independent (notebook 08)', 'CPTAC lung': 'one collection per class (notebook 08)',
        'NSCLC-Radiomics': 'independent (new, plan 7j)', 'Lung-PET-CT-Dx': 'independent (new, plan 7j)'}
# one class order for every dataset: (class 0, class 1); class 1 is squamous cell carcinoma
CLASSES = {SOURCE: ('TCGA-LUAD', 'TCGA-LUSC'), 'NSCLC Radiogenomics': ('adenocarcinoma', 'squamous'),
           'CPTAC lung': ('CPTAC-LUAD', 'CPTAC-LSCC'),
           'NSCLC-Radiomics': ('adenocarcinoma', 'squamous cell carcinoma'), 'Lung-PET-CT-Dx': ('A', 'G')}
# counts written into the plan before any image (old targets: notebook 08's plan; new targets: plan 7j A.1)
EXPECTED_TARGETS = {'NSCLC Radiogenomics': {'adenocarcinoma': 166, 'squamous': 33},
                    'CPTAC lung': {'CPTAC-LUAD': 37, 'CPTAC-LSCC': 36}}
EXPECTED_LABELS_NEW = {'NSCLC-Radiomics': {'adenocarcinoma': 51, 'squamous cell carcinoma': 152},
                       'Lung-PET-CT-Dx': {'A': 251, 'G': 61}}
METADATA_TARGETS = ()       # the metadata model is not part of 7j (the new cohorts were not in notebook 02)

# Lung1 clinical file (NSCLC-Radiomics), pinned on 3 Nov 2026
LUNG1_COLLECTION = 'NSCLC-Radiomics'
LUNG1_URL = ('https://www.cancerimagingarchive.net/wp-content/uploads/'
             'NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv')
LUNG1_MD5 = '2d2638a2ac1d34edc6535b59b6f71510'
# Lung-PET-CT-Dx: the class is the letter after "Lung_Dx-" (A adenocarcinoma, B small cell, E large cell, G squamous)
DX_COLLECTION = 'Lung-PET-CT-Dx'
DX_ID = r'Lung_Dx-([ABEG])\d{4}'

# Plan 7j A.6 (descriptive): the fields compared by class in every lung dataset (notebook 09's list)
FIELDS_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_recon_diameter', 'hdr_kvp', 'hdr_tube_current',
                  'body_area_cm2']

# Notebook 01's rule 7a.7 (frozen settings, copied unchanged; the function below is checked against notebook 01's output)
IMAGING_MODALITIES = ('CT', 'MR')
MIN_IMAGES = 10
EXCLUDE_SERIES_PATTERN = (r'scout|locali[sz]er|\bloc\b|topogram|survey|calibration|\bcal\b|'
                          r'bolus|smart\s*prep|monitor|dose\s*(?:report|info)|screen\s*save|'
                          r'secondary|derived|\bmpr\b|\bmip\b|reformat|fusion')
CT_REFORMAT_PATTERN = r'\bsag|\bcor\b|coronal'

# Local code tests only (ignored in Colab)
TEST = {}
if not IN_COLAB:
    TEST = {'patients': [p for p in os.environ.get('ASA_TEST_PATIENTS', '').split(';') if p],
            'fake_images': os.environ.get('ASA_TEST_FAKE_IMAGES') == '1',
            'stop_after_images': os.environ.get('ASA_TEST_STOP_AFTER_IMAGES') == '1',
            'real_is_shuffled': True}   # local tests never see the real labels
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    CTFM_BATCH = int(os.environ.get('ASA_TEST_CTFM_BATCH', CTFM_BATCH))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [4]:
# 4. Load and verify the inputs from notebooks 01-09; the source and the four lung targets
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder} changed since that run: {bad}'
    return man

def cache_md5(folder):
    files = sorted((q for q in folder.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv')),
                   key=lambda q: q.relative_to(folder).parts)
    return len(files), hashlib.md5(''.join(md5(q) for q in files).encode()).hexdigest()

man01, man02, man03, man04 = verify(IN01), verify(IN02), verify(IN03), verify(IN04)
man = {nb: {m: verify(d / m) for m in ('rehearsal', 'real')} for nb, d in (('05', IN05), ('06', IN06), ('07', IN07),
                                                                          ('08', IN08), ('09', IN09))}
runs = {'01': man01['run_started_utc'], '02': man02['run_started_utc'], '03': man03['run_started_utc'],
        '04': man04['run_started_utc']}
for nb in ('05', '06', '07', '08', '09'):
    for m in ('rehearsal', 'real'):
        assert man[nb][m]['input_runs'] == runs, f'notebook {nb} ({m}) was run on other inputs'
        if not TEST:
            assert man[nb][m]['in_colab'] and not man[nb][m]['local_test_settings'], f'notebook {nb} {m} is not the Colab run'
    runs = {**runs, nb: man[nb]['real']['run_started_utc']}
for nb in ('05', '07', '08'):
    assert man[nb]['real']['labels'] == 'real' and man[nb]['rehearsal']['labels'].startswith('shuffled'), nb
s05 = man['05']['real']['settings']
same = {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'MIN_COUNT': MIN_COUNT,
        'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'LR_PARAMS': LR_PARAMS, 'BODY_HU': BODY_HU,
        'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU, 'HDR_COLS': HDR_COLS, 'PREP_KEY': PREP_KEY05,
        'EMB_KEY': EMB_KEY05, 'CTFM': [CTFM_REPO, CTFM_REVISION, list(CTFM_SPACING), list(CTFM_PATCH), CTFM_OVERLAP],
        'MERLIN': [MERLIN_REPO, MERLIN_REVISION, MERLIN_CKPT]}
if not TEST:
    same.update({'N_BOOT': N_BOOT, 'N_PERM': N_PERM})
diff = [k for k, v in same.items() if json.loads(json.dumps(s05[k])) != json.loads(json.dumps(v))]
assert not diff, f'settings differ from notebook 05: {diff}'
s08 = man['08']['real']['settings']
assert (s08['SIL_HU'], s08['EMB_KEY08'], s08['PREP_KEY08']) == (SIL_HU, EMB_KEY, PREP_KEY), 'settings differ from notebook 08'
assert list(s08['INPUTS_B']) == list(INPUTS_B), 'notebook 08 analysed other inputs'
assert all(man['08']['real']['classes'][k] == list(CLASSES[k]) for k in (SOURCE,) + OLD_TARGETS), \
    'class order differs from notebook 08'
caches = {}
for nb, folder in (('05', CACHE05), ('06', CACHE06), ('07', CACHE07), ('08', CACHE08)):
    n, h = cache_md5(folder)
    assert (n, h) == (man[nb]['real']['cache_files']['count'], man[nb]['real']['cache_files']['combined_md5']), \
        f'notebook {nb} cache differs from its run record'
    caches[nb] = {'count': n, 'combined_md5': h}
v05 = man['05']['real']['versions']
same_versions = (v05['numpy'], v05['pandas'], v05['scipy'], v05['scikit-learn']) == \
                (np.__version__, pd.__version__, scipy.__version__, sklearn.__version__)
print('Inputs verified: notebooks 01-09 and the caches of notebooks 05-08 '
      f"({', '.join(f'{k}: {v['count']} files' for k, v in caches.items())})")
print('Package versions', 'match notebook 05' if same_versions else
      f"DIFFER from notebook 05 (then numpy {v05['numpy']}, pandas {v05['pandas']}, scipy {v05['scipy']}, "
      f"scikit-learn {v05['scikit-learn']}): small differences in the fitted models are possible")

labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
main = pd.read_csv(IN01 / 'main_series.csv', low_memory=False)
smeta = pd.read_csv(IN01 / 'series_metadata.csv.gz', usecols=['SeriesInstanceUID', 'FileSize']).drop_duplicates(
    'SeriesInstanceUID')
KEY = ['collection', 'PatientID', 'modality']
text = (toks0.assign(group=np.where(toks0.category == 'care pathway', 'care_text', 'protocol_text'))
             .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat0.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
POLISH_WORDS = set(man04['word_lists']['POLISH_WORDS'])
polish = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)
a03 = pd.read_csv(IN03 / 'audit_primary.csv').set_index('task_id')
assert a03.flagged[SOURCE] and SOURCE in man['05']['real']['tasks'], 'source task changed'

inv = {nb: pd.read_csv(folder / 'image_inventory.csv') for nb, folder in (('05', CACHE05), ('07', CACHE07),
                                                                            ('08', CACHE08))}
IN05_KEYS, IN07_KEYS, IN08_KEYS = (set(inv[nb][inv[nb].embedded].key) for nb in ('05', '07', '08'))

def main_series_rule(series):
    """One main series per patient and modality: the earliest study; in it, the eligible series with the most images."""
    series = series.replace(r'^\s*$', np.nan, regex=True)
    series['ImageCount'] = pd.to_numeric(series['ImageCount'], errors='coerce')
    series['StudyDate_dt'] = pd.to_datetime(series['StudyDate'], errors='coerce')
    img = series[series.Modality.isin(IMAGING_MODALITIES)].copy()
    desc = img.SeriesDescription.where(img.SeriesDescription.notna(), img.ProtocolName).fillna('')
    img['excluded_pattern'] = desc.str.contains(EXCLUDE_SERIES_PATTERN, flags=re.I, regex=True)
    img['excluded_ct_reformat'] = (img.Modality == 'CT') & desc.str.contains(CT_REFORMAT_PATTERN, flags=re.I, regex=True)
    img['excluded_third_party'] = (img.ThirdPartyAnalysis.astype(str).str.strip().str.lower() == 'yes'
                                   if 'ThirdPartyAnalysis' in img else False)
    img['excluded_small'] = img.ImageCount.fillna(0) < MIN_IMAGES
    img['eligible'] = ~(img.excluded_pattern | img.excluded_ct_reformat | img.excluded_third_party | img.excluded_small)
    pts_by_mod = img.groupby(['Collection', 'Modality']).PatientID.nunique().unstack(fill_value=0)
    primary_modality = {c: ('CT' if r.get('CT', 0) >= r.get('MR', 0) else 'MR') for c, r in pts_by_mod.iterrows()}
    main_rows, no_main = [], []
    for (col, mod, pid), g in img.groupby(['Collection', 'Modality', 'PatientID'], sort=True):
        order = (g.groupby('StudyInstanceUID').StudyDate_dt.min().reset_index()
                  .sort_values(['StudyDate_dt', 'StudyInstanceUID'], na_position='last'))
        for rank, suid in enumerate(order.StudyInstanceUID, 1):
            e = g[(g.StudyInstanceUID == suid) & g.eligible]
            if len(e):
                b = e.sort_values(['ImageCount', 'SeriesNumber', 'SeriesInstanceUID'],
                                  ascending=[False, True, True]).iloc[0]
                main_rows.append({'collection': col, 'PatientID': pid, 'modality': mod,
                                  'is_primary_modality': mod == primary_modality[col],
                                  'StudyInstanceUID': suid, 'StudyDate': b.StudyDate,
                                  'study_rank_used': rank, 'n_studies_this_modality': len(order),
                                  'n_series_this_modality': len(g),
                                  'SeriesInstanceUID': b.SeriesInstanceUID, 'SeriesNumber': b.SeriesNumber,
                                  'SeriesDescription': b.SeriesDescription, 'ProtocolName': b.ProtocolName,
                                  'Manufacturer': b.Manufacturer, 'ManufacturerModelName': b.ManufacturerModelName,
                                  'SoftwareVersions': b.SoftwareVersions, 'ImageCount': b.ImageCount})
                break
        else:
            no_main.append({'collection': col, 'PatientID': pid, 'modality': mod})
    return pd.DataFrame(main_rows), pd.DataFrame(no_main, columns=['collection', 'PatientID', 'modality'])

check, _ = main_series_rule(pd.read_csv(IN01 / 'series_metadata.csv.gz', low_memory=False))
cmp = main[KEY + ['SeriesInstanceUID']].merge(check[KEY + ['SeriesInstanceUID']], on=KEY, how='outer',
                                              suffixes=('', '_rule'))
RULE_REPRODUCED = bool(len(check) == len(main) and (cmp.SeriesInstanceUID == cmp.SeriesInstanceUID_rule).all())
assert RULE_REPRODUCED, 'the main-series function does not reproduce notebook 01'
print(f'Main-series rule: reproduces notebook 01 for all {len(main):,} patient-modality pairs')

def tcia_json(endpoint, **params):
    """A TCIA API reply (series metadata, no images), cached on Drive so that reruns use the same reply."""
    path = CACHE / 'api' / f"{endpoint}__{'__'.join(f'{k}={v}' for k, v in sorted(params.items()))}.json"
    if not path.exists():
        last = None
        for wait in (0, 10, 30, 90):
            time.sleep(wait)
            try:
                r = requests.get(f'{TCIA}/{endpoint}', params=params, timeout=(30, 300))
                r.raise_for_status()
                data = r.json()
                assert len(data), 'empty reply'
                write_json_atomic(path, {'request': {'endpoint': endpoint, **params}, 'fetched_utc':
                                         datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                                         'data': data})
                break
            except Exception as e:
                last = e
        else:
            raise RuntimeError(f'TCIA {endpoint} failed 4 times: {last!r}')
    return json.loads(path.read_text())['data'], md5(path)

# old target 1: NSCLC Radiogenomics (notebook 02's task; embedded in notebook 07), as notebook 08 built it
nsclc = labels[labels.task_id == NSCLC_TASK].copy()
nsclc['key'] = nsclc.collection + '__' + nsclc.PatientID
# old target 2: CPTAC lung (every patient with a CT main series in notebook 01; the class is the collection)
cptac = main[main.collection.isin(CLASSES['CPTAC lung']) & (main.modality == 'CT')].copy()
assert not cptac.PatientID.duplicated().any(), 'a patient is in both CPTAC lung collections'
cptac['label'] = cptac.collection
cptac = cptac.merge(smeta, on='SeriesInstanceUID', how='left')

# new target 1: NSCLC-Radiomics, labels from the pinned Lung1 clinical file
lung1_path = LABEL_DIR / 'NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv'
if not lung1_path.exists():
    r = requests.get(LUNG1_URL, timeout=(30, 120))
    r.raise_for_status()
    lung1_path.write_bytes(r.content)
assert md5(lung1_path) == LUNG1_MD5, 'the Lung1 clinical file differs from the pinned one'
lung1 = pd.read_csv(lung1_path)
lung1 = lung1[lung1.Histology.isin(CLASSES['NSCLC-Radiomics'])][['PatientID', 'Histology']].rename(
    columns={'Histology': 'label'})
# new target 2: Lung-PET-CT-Dx, the class from the patient-ID code
NEW_FRAMES, API_MD5, NO_SERIES, SERIES_NEW = {}, {}, {}, {}
for coll in (LUNG1_COLLECTION, DX_COLLECTION):
    reply, API_MD5[coll] = tcia_json('getSeries', Collection=coll)
    series = pd.DataFrame(reply)
    SERIES_NEW[coll] = series            # kept for the amendment of 4 Nov 2026 (series without CT slices)
    m_ct, _ = main_series_rule(series)
    m_ct = m_ct[m_ct.modality == 'CT']
    sizes = series.drop_duplicates('SeriesInstanceUID').set_index('SeriesInstanceUID').FileSize.astype(float)
    if coll == LUNG1_COLLECTION:
        lab = lung1
    else:
        ids = pd.Series(series.PatientID.unique())
        assert ids.str.fullmatch(DX_ID).all(), 'a Lung-PET-CT-Dx patient ID does not follow the documented code'
        lab = pd.DataFrame({'PatientID': ids, 'label': ids.str.extract(DX_ID, expand=False)})
        lab = lab[lab.label.isin(CLASSES['Lung-PET-CT-Dx'])]
    name = 'NSCLC-Radiomics' if coll == LUNG1_COLLECTION else 'Lung-PET-CT-Dx'
    counts = lab.label.value_counts().to_dict()
    assert counts == EXPECTED_LABELS_NEW[name], f'{name}: label counts {counts} differ from plan 7j {EXPECTED_LABELS_NEW[name]}'
    df = lab.merge(m_ct, on='PatientID', how='left')
    NO_SERIES[name] = df[df.SeriesInstanceUID.isna()].PatientID.tolist()
    df = df[df.SeriesInstanceUID.notna()].copy()
    df['collection'] = coll
    df['FileSize'] = df.SeriesInstanceUID.map(sizes)
    NEW_FRAMES[name] = df

TARGETS = {}
for name, df in (('NSCLC Radiogenomics', nsclc), ('CPTAC lung', cptac), *NEW_FRAMES.items()):
    df = df.copy()
    df['key'] = df.collection + '__' + df.PatientID
    df['modality'] = 'CT'
    assert set(df.label) <= set(CLASSES[name]), f'{name}: unexpected labels {sorted(set(df.label))}'
    if name in EXPECTED_TARGETS:
        counts = df.label.value_counts().to_dict()
        assert counts == EXPECTED_TARGETS[name], f'{name}: {counts} differs from notebook 08 {EXPECTED_TARGETS[name]}'
    TARGETS[name] = df.sort_values('key').reset_index(drop=True)
assert set(TARGETS['NSCLC Radiogenomics'].key) <= IN07_KEYS, 'an NSCLC Radiogenomics patient is not embedded in notebook 07'
assert set(TARGETS['CPTAC lung'].key) <= IN08_KEYS, 'a CPTAC lung patient is not embedded in notebook 08'

new = pd.concat([TARGETS[t].assign(target=t) for t in NEW_TARGETS], ignore_index=True)
assert not new.key.isin(IN05_KEYS | IN07_KEYS | IN08_KEYS).any(), 'a new target patient was already embedded'
PATIENTS = new.sort_values('key').reset_index(drop=True)
NEW = PATIENTS
if TEST.get('patients'):
    NEW = NEW[NEW.key.isin(TEST['patients'])].reset_index(drop=True)
QC_KEYS = set(NEW.sort_values('key').groupby('target').key.head(N_QC))
print('Targets: ' + '; '.join(f"{t} {len(d)} ({' / '.join(f'{c} {n}' for c, n in d.label.value_counts().items())})"
                              for t, d in TARGETS.items()))
print('New-target patients without an eligible CT main series (dropped, reported):',
      {k: len(v) for k, v in NO_SERIES.items()})
print(f'{len(PATIENTS)} new patients ({PATIENTS.FileSize.sum() / 1e9:.1f} GB of DICOM to download)')

Inputs verified: notebooks 01-09 and the caches of notebooks 05-08 (05: 1053 files, 06: 1052 files, 07: 1450 files, 08: 477 files)
Package versions DIFFER from notebook 05 (then numpy 2.1.3, pandas 2.2.3, scipy 1.16.3, scikit-learn 1.6.1): small differences in the fitted models are possible
Main-series rule: reproduces notebook 01 for all 4,883 patient-modality pairs
Targets: NSCLC Radiogenomics 199 (adenocarcinoma 166 / squamous 33); CPTAC lung 73 (CPTAC-LUAD 37 / CPTAC-LSCC 36); NSCLC-Radiomics 203 (squamous cell carcinoma 152 / adenocarcinoma 51); Lung-PET-CT-Dx 312 (A 251 / G 61)
New-target patients without an eligible CT main series (dropped, reported): {'NSCLC-Radiomics': 0, 'Lung-PET-CT-Dx': 0}
515 new patients (50.0 GB of DICOM to download)


## Part 1 · Download and prepare the new scans (no labels used)
Same series rule, reading, body mask, background-only version and silhouette as notebooks 05–08. Each new scan's
DICOM-header fields are saved like notebook 05's. Scans embedded in notebooks 05, 07 and 08 are not downloaded again.

In [5]:
# 5. Download, read and convert one series (notebook 08, unchanged)
def download_series(uid, dest):
    last = None
    for wait in (0, 10, 30, 90, 240):
        time.sleep(wait)
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(f'{TCIA}/getImage', params={'SeriesInstanceUID': uid}, stream=True,
                              timeout=(30, 600)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(tmp) as z:
                badf = z.testzip()
                assert badf is None, f'corrupt file in zip: {badf}'
            tmp.replace(dest)
            return dest.stat().st_size
        except Exception as e:   # network errors, partial files: try again
            last = e
    raise RuntimeError(f'download failed 5 times: {last!r}')

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def _text(v):
    if v is None:
        return np.nan
    if isinstance(v, pydicom.multival.MultiValue):
        v = '/'.join(str(x) for x in v)
    v = str(v).strip().upper()
    return v if v else np.nan

def read_series(folder, uid):
    """The slices of one series, sorted along the slice normal, with geometry checks."""
    metas = []
    for p in sorted(folder.rglob('*')):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
        except Exception:
            continue
        if str(getattr(ds, 'SeriesInstanceUID', '')) != uid:
            continue
        if not all(t in ds for t in ('ImagePositionPatient', 'ImageOrientationPatient', 'PixelSpacing', 'Rows')):
            continue
        metas.append((p, ds))
    assert metas, 'no image slices in the download'
    combo = [(tuple(np.round(np.asarray(ds.ImageOrientationPatient, float), 3)), int(ds.Rows), int(ds.Columns))
             for _, ds in metas]
    major = pd.Series(combo).value_counts().index[0]
    keep = [m for m, c in zip(metas, combo) if c == major]
    iop = np.asarray(keep[0][1].ImageOrientationPatient, float)
    row_cos, col_cos = iop[:3], iop[3:]
    normal = np.cross(row_cos, col_cos)
    pos = np.array([[float(v) for v in ds.ImagePositionPatient] for _, ds in keep])
    proj = pos @ normal
    inst = np.array([_num(getattr(ds, 'InstanceNumber', np.nan)) for _, ds in keep])
    order = np.lexsort((np.nan_to_num(inst, nan=1e9), proj))
    dup = np.r_[False, np.diff(proj[order]) < 1e-3]          # same position twice: keep the first
    order = order[~dup]
    keep, pos, proj = [keep[i] for i in order], pos[order], proj[order]
    assert len(keep) >= 10, f'only {len(keep)} usable slices'
    gaps = np.diff(proj)
    z = float(np.median(gaps))
    step = pos[-1] - pos[0]
    geo = {'row_cos': row_cos, 'col_cos': col_cos, 'normal': normal, 'origin': pos[0], 'z': z,
           'n_other_orientation_dropped': sum(c != major for c in combo),
           'n_duplicates_dropped': int(dup.sum()),
           'max_gap_deviation_mm': float(np.max(np.abs(gaps - z))),
           'nonuniform_spacing': bool(np.max(np.abs(gaps - z)) > max(0.05, 0.1 * z)),
           'gantry_tilt_deg': float(np.degrees(np.arccos(min(1.0, abs(step @ normal) / np.linalg.norm(step))))),
           'coverage_mm': float(proj[-1] - proj[0] + z)}
    return keep, geo

def header_fields(keep, geo):
    ds0 = keep[0][1]
    def med(name):
        vals = [_num(getattr(ds, name, None)) for _, ds in keep]
        vals = [v for v in vals if np.isfinite(v)]
        return float(np.median(vals)) if vals else np.nan
    ps = [float(v) for v in ds0.PixelSpacing]
    agent = getattr(ds0, 'ContrastBolusAgent', None)
    return {'hdr_manufacturer': _text(getattr(ds0, 'Manufacturer', None)),
            'hdr_model': _text(getattr(ds0, 'ManufacturerModelName', None)),
            'hdr_slice_thickness': med('SliceThickness'), 'hdr_z_spacing': geo['z'],
            'hdr_pixel_spacing': float(np.mean(ps)), 'hdr_rows': int(ds0.Rows), 'hdr_columns': int(ds0.Columns),
            'hdr_n_slices': len(keep), 'hdr_coverage_mm': geo['coverage_mm'],
            'hdr_recon_diameter': med('ReconstructionDiameter'), 'hdr_kvp': med('KVP'),
            'hdr_tube_current': med('XRayTubeCurrent'), 'hdr_exposure': med('Exposure'),
            'hdr_ctdivol': med('CTDIvol'), 'hdr_pitch': med('SpiralPitchFactor'),
            'hdr_kernel': _text(getattr(ds0, 'ConvolutionKernel', None)),
            'hdr_filter_type': _text(getattr(ds0, 'FilterType', None)),
            'hdr_patient_position': _text(getattr(ds0, 'PatientPosition', None)),
            'hdr_contrast_agent_given': ('yes' if agent is not None and str(agent).strip() else 'no'),
            'hdr_gantry_tilt_deg': geo['gantry_tilt_deg'], 'hdr_nonuniform_spacing': geo['nonuniform_spacing'],
            'hdr_max_gap_deviation_mm': geo['max_gap_deviation_mm'],
            'hdr_duplicates_dropped': geo['n_duplicates_dropped'],
            'hdr_other_orientation_dropped': geo['n_other_orientation_dropped']}

def to_image(keep, geo):
    """HU volume with the geometry from the DICOM positions (not from the reader's guess)."""
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(p) for p, _ in keep])
    reader.SetOutputPixelType(sitk.sitkFloat32)          # GDCM applies the rescale slope and intercept (HU)
    img = reader.Execute()
    ps = [float(v) for v in keep[0][1].PixelSpacing]      # [between rows, between columns]
    img.SetSpacing((ps[1], ps[0], geo['z']))
    img.SetOrigin(tuple(float(v) for v in geo['origin']))
    img.SetDirection(tuple(np.stack([geo['row_cos'], geo['col_cos'], geo['normal']], axis=1).ravel()))
    return img

def like(arr, ref, dtype=np.int16):
    """A SimpleITK image from a (z, y, x) array, with the geometry of `ref`."""
    if dtype == np.int16:
        arr = np.clip(np.round(arr), -32768, 32767)
    out = sitk.GetImageFromArray(arr.astype(dtype))
    out.CopyInformation(ref)
    return out

In [6]:
# 6. Body mask, background-only version, silhouette and QC picture (notebook 08, unchanged)
def body_mask(arr, sp):
    """arr: (z, y, x) HU; sp: (sz, sy, sx) mm. Returns (body, body dilated by DILATE_MM)."""
    m = arr > BODY_HU
    r = max(1, int(round(OPEN_MM / ((sp[1] + sp[2]) / 2))))
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    m = ndimage.binary_opening(m, structure=(yy ** 2 + xx ** 2 <= r * r)[None])
    lab_, n = ndimage.label(m)
    if n == 0:
        return np.zeros_like(m), np.zeros_like(m)
    counts = np.bincount(lab_.ravel())
    counts[0] = 0
    largest = lab_ == counts.argmax()
    del lab_, m
    core = ndimage.binary_fill_holes(largest, structure=np.ones((1, 3, 3), bool))
    del largest
    # dilation by DILATE_MM, computed inside the bounding box plus a margin (same result, much faster)
    pad = [int(np.ceil(DILATE_MM / s)) + 1 for s in sp]
    idx = np.where(core)
    lo = [max(0, int(i.min()) - p) for i, p in zip(idx, pad)]
    hi = [min(n_, int(i.max()) + p + 1) for i, n_, p in zip(idx, core.shape, pad)]
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    dil = np.zeros_like(core)
    dil[box] = ndimage.distance_transform_edt(~core[box], sampling=sp).astype(np.float32) <= DILATE_MM
    return core, dil

def silhouette(dil):
    """Notebook 06: the region the background-only version blanked, as a flat shape; air outside."""
    return np.where(dil, SIL_HU, AIR_HU).astype(np.int16)

def qc_picture(key, arr, core, bg, sil, path):
    from matplotlib.figure import Figure
    z = int(np.argmax(core.sum(axis=(1, 2))))
    fig = Figure(figsize=(12, 4.2))
    ax = fig.subplots(1, 3)
    kw = dict(cmap='gray', vmin=-1000, vmax=400)
    ax[0].imshow(arr[z], **kw); ax[0].contour(core[z], levels=[0.5], colors='r', linewidths=0.6)
    ax[0].set_title('original + body outline')
    ax[1].imshow(bg[z], **kw); ax[1].set_title('background only')
    ax[2].imshow(sil[z], **kw); ax[2].set_title('silhouette')
    for a in ax:
        a.axis('off')
    fig.suptitle(key, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=80)

def nifti_paths(key):
    d = WORK / 'nifti'
    return {v: d / f'{key}__{v}.nii.gz' for v in VARIANTS}

def header_path(key):
    return CACHE / 'headers' / f'{key}.json'

def header_ok(key):
    p = header_path(key)
    return p.exists() and json.loads(p.read_text()).get('prep_key') == PREP_KEY

In [7]:
# 7. Prepare one new patient (runs in parallel workers) (notebook 08, unchanged)
def prepare_patient(key, uid, file_size):
    paths = nifti_paths(key)
    if header_ok(key) and all(p.exists() for p in paths.values()):
        return {'key': key, 'status': 'ready'}
    t0 = time.time()
    tmp = WORK / 'dicom' / key
    zpath = WORK / 'dicom' / f'{key}.zip'
    try:
        shutil.rmtree(tmp, ignore_errors=True)
        nbytes = download_series(uid, zpath)
        with zipfile.ZipFile(zpath) as z:
            n_files = len([n for n in z.namelist() if not n.endswith('/')])
            z.extractall(tmp)
        keep, geo = read_series(tmp, uid)
        hdr = header_fields(keep, geo)
        img = to_image(keep, geo)
        arr = sitk.GetArrayFromImage(img)
        sp = img.GetSpacing()[::-1]
        core, dil = body_mask(arr, sp)
        bg = arr.copy()
        bg[dil] = AIR_HU
        sil = silhouette(dil)
        for name, a in (('original', arr), ('background', bg), ('silhouette', sil)):
            sitk.WriteImage(like(a, img, np.int16), str(paths[name]), useCompression=True)
        area = core.sum(axis=(1, 2)) * sp[1] * sp[2] / 100.0
        sub = arr[::2, ::4, ::4]
        rec = {'key': key, 'SeriesInstanceUID': uid, 'prep_key': PREP_KEY, 'download_bytes': nbytes,
               'tcia_file_size': file_size, 'n_files_in_zip': n_files, **hdr,
               'body_area_cm2': float(area[area > 0].mean()) if (area > 0).any() else np.nan,
               'body_fraction_dilated': float(dil.mean()),
               'hu_p01': float(np.percentile(sub, 1)), 'hu_p50': float(np.percentile(sub, 50)),
               'hu_p99': float(np.percentile(sub, 99)), 'prep_seconds': round(time.time() - t0, 1)}
        if key in QC_KEYS:
            qc_picture(key, arr, core, bg, sil, CACHE / 'qc' / f'{key}.png')
        write_json_atomic(header_path(key), rec)
        return {'key': key, 'status': 'ok', 'seconds': rec['prep_seconds']}
    except Exception as e:
        return {'key': key, 'status': 'failed', 'error': repr(e)[:400]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
        zpath.unlink(missing_ok=True)

### Encoders (notebook 05, unchanged)

In [8]:
# 8. Load the two encoders (notebook 08, unchanged)
i3res.checkpoint = type('NoCheckpoint', (), {'checkpoint': staticmethod(lambda fn, *a, **k: fn(*a))})

def load_ctfm():
    cfg = json.loads(Path(hf_hub_download(CTFM_REPO, 'config.json', revision=CTFM_REVISION)).read_text())
    model = SegResEncoder(**cfg)
    state = safetensors.torch.load_file(hf_hub_download(CTFM_REPO, 'model.safetensors', revision=CTFM_REVISION))
    model.load_state_dict(state, strict=True)
    return model.eval()

MERLIN_DIR = Path(merlin.__file__).parent / 'models' / 'checkpoints'   # where the official class looks for it
if not IN_COLAB and os.environ.get('ASA_TEST_MERLIN_DIR'):              # local tests: Windows path-length limit
    MERLIN_DIR = Path(os.environ['ASA_TEST_MERLIN_DIR'])

def load_merlin():
    ckpt = hf_hub_download(MERLIN_REPO, MERLIN_CKPT, revision=MERLIN_REVISION, local_dir=MERLIN_DIR)
    state = torch.load(ckpt, map_location='cpu')
    prefix = 'encode_image.i3_resnet.'
    enc = i3res.I3ResNet(copy.deepcopy(torchvision.models.resnet152(weights=None)), class_nb=1692, conv_class=True,
                         ImageEmbedding=True)
    enc.load_state_dict({k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}, strict=True)
    return enc.eval()

# CT-FM preprocessing (authors' script), split so that the background-only version can use the original's crop box
CTFM_LOAD = Compose([LoadImage(ensure_channel_first=True), EnsureType(), Orientation(axcodes='SPL'),
                     Spacing(pixdim=CTFM_SPACING, mode='bilinear')])
CTFM_SCALE = ScaleIntensityRange(a_min=-1024, a_max=2048, b_min=0, b_max=1, clip=True)

def ctfm_inputs(paths, crop_from=None):
    """Preprocessed CT-FM inputs for each variant. With crop_from, every variant uses that variant's foreground box."""
    imgs = {v: CTFM_LOAD(str(p)) for v, p in paths.items()}
    out, boxes = {}, {}
    for v, img in imgs.items():
        src = imgs[crop_from] if crop_from else img
        start, end = generate_spatial_bounding_box(src, select_fn=lambda x: x > 0)   # CropForeground's default
        out[v] = _plain(CTFM_SCALE(SpatialCrop(roi_start=start, roi_end=end)(img)))
        boxes[v] = [int(a) for a in start] + [int(b) for b in end]
    return out, boxes

def _plain(t):
    return (t.as_tensor() if hasattr(t, 'as_tensor') else torch.as_tensor(t)).float()

def merlin_inputs(paths):
    return {v: _plain(MERLIN_TRANSFORMS({'image': str(p)})['image']) for v, p in paths.items()}

class PrepDataset(torch.utils.data.Dataset):
    """CPU preprocessing in DataLoader workers; the GPU only runs the encoders."""
    def __init__(self, items):
        self.items = items              # (key, {variant: path}, crop_from)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        key, paths, crop_from = self.items[i]
        try:
            ct, boxes = ctfm_inputs(paths, crop_from)
            return {'key': key, 'ctfm': ct, 'merlin': merlin_inputs(paths), 'boxes': boxes, 'error': None}
        except Exception as e:
            root = e
            while root.__cause__ is not None:      # MONAI wraps the real error; report the root cause
                root = root.__cause__
            return {'key': key, 'error': f'{e!r} | root cause: {root!r}'[:600]}

@torch.no_grad()
def ctfm_embed(model, vol):
    feats, batch = [], []
    def run(b):
        x = torch.cat(b).to(DEVICE)
        return torch.nn.functional.adaptive_avg_pool3d(model(x)[-1], 1).flatten(1).cpu()
    for patch, _ in SlidingWindowSplitter(CTFM_PATCH, CTFM_OVERLAP)(vol.unsqueeze(0)):
        batch.append(patch)
        if len(batch) == CTFM_BATCH:
            feats.append(run(batch)); batch = []
    if batch:
        feats.append(run(batch))
    f = torch.cat(feats)
    return f.mean(0).numpy().astype(np.float32), int(len(f))

@torch.no_grad()
def merlin_embed(model, vol):
    return model(vol.unsqueeze(0).to(DEVICE)).reshape(-1).cpu().numpy().astype(np.float32)

def embed_items(items, out_path_of, models, extra_of=None, label='scans'):
    """Embed every item with both encoders and save one .npz per item (resume point)."""
    loader = torch.utils.data.DataLoader(PrepDataset(items), batch_size=None, shuffle=False,
                                         num_workers=EMB_WORKERS if len(items) > 1 else 0)
    done, failed, t0 = 0, [], time.time()
    for rec in loader:
        if rec['error']:
            failed.append({'key': rec['key'], 'error': rec['error']})
            continue
        arrays, meta = {}, {'emb_key': EMB_KEY, 'boxes': rec['boxes'], **(extra_of(rec['key']) if extra_of else {})}
        for v, vol in rec['ctfm'].items():
            arrays[f'ctfm__{v}'], meta[f'ctfm__{v}__patches'] = ctfm_embed(models['ctfm'], vol)
        for v, vol in rec['merlin'].items():
            arrays[f'merlin__{v}'] = merlin_embed(models['merlin'], vol)
        path = out_path_of(rec['key'])
        tmp = path.with_name(path.stem + '.tmp.npz')
        np.savez(tmp, meta=json.dumps(meta, default=_jsonable), **arrays)
        tmp.replace(path)
        done += 1
        if done % 25 == 0 or done == len(items):
            print(f'  {label}: {done}/{len(items)} embedded ({(time.time() - t0) / 60:.1f} min)')
    return failed

def emb_path(key):
    return CACHE / 'embeddings' / f'{key}.npz'

def emb_ok(path, variants):
    if not path.exists():
        return False
    with np.load(path) as z:
        meta = json.loads(str(z['meta']))
        return meta.get('emb_key') == EMB_KEY and all(f'{e}__{v}' in z.files for e in ENCODERS for v in variants)

def emb_file(key, variant):
    """Where a scan's embedding is: notebook 05 (original, background) or 06 (silhouette), notebook 07, or this notebook."""
    if key in IN05_KEYS:
        return (CACHE06 if variant == 'silhouette' else CACHE05) / 'embeddings' / f'{key}.npz'
    if key in IN07_KEYS:
        return CACHE07 / 'embeddings' / f'{key}.npz'
    return emb_path(key)

def has_variant(key, variant):
    p = emb_file(key, variant)
    if not p.exists():
        return False
    with np.load(p) as z:
        ok_key = json.loads(str(z['meta'])).get('emb_key') in (EMB_KEY05, EMB_KEY)
        return ok_key and all(f'{e}__{variant}' in z.files for e in ENCODERS)

def header_of(key):
    folder = CACHE05 if key in IN05_KEYS else CACHE07 if key in IN07_KEYS else CACHE
    p = folder / 'headers' / f'{key}.json'
    return json.loads(p.read_text()) if p.exists() else {'key': key}

/usr/local/lib/python3.13/dist-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.array Orientation.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [ ]:
# 9. Part 1: prepare and embed the new scans (label-free; resumable)
def emb_file(key, variant):
    """Where a scan's embedding is: notebook 05 (original, background) or 06 (silhouette), notebook 07, 08, or this one."""
    if key in IN05_KEYS:
        return (CACHE06 if variant == 'silhouette' else CACHE05) / 'embeddings' / f'{key}.npz'
    if key in IN07_KEYS:
        return CACHE07 / 'embeddings' / f'{key}.npz'
    if key in IN08_KEYS:
        return CACHE08 / 'embeddings' / f'{key}.npz'
    return emb_path(key)

def has_variant(key, variant):
    p = emb_file(key, variant)
    if not p.exists():
        return False
    with np.load(p) as z:
        ok_key = json.loads(str(z['meta'])).get('emb_key') in (EMB_KEY05, EMB_KEY)
        return ok_key and all(f'{e}__{variant}' in z.files for e in ENCODERS)

def header_of(key):
    folder = CACHE05 if key in IN05_KEYS else CACHE07 if key in IN07_KEYS else CACHE08 if key in IN08_KEYS else CACHE
    p = folder / 'headers' / f'{key}.json'
    return json.loads(p.read_text()) if p.exists() else {'key': key}

# Amendment of 4 Nov 2026 (label-blind): a series the rule picks but that holds no CT slices is set aside, and the rule
# picks again from the patient's other series (notebook 08's prepare_patient itself is unchanged)
NO_SLICES = 'no image slices in the download'
SERIES_COLS = ['StudyInstanceUID', 'StudyDate', 'SeriesInstanceUID', 'SeriesNumber', 'SeriesDescription', 'ProtocolName',
               'Manufacturer', 'ManufacturerModelName', 'SoftwareVersions', 'ImageCount', 'FileSize']

def rule_main_without(pid_series, set_aside):
    """The CT series notebook 01's rule picks for one patient once the series in set_aside are left out (None if no
    eligible CT series is left). pid_series: that patient's rows of the getSeries reply."""
    s = pid_series[(pid_series.Modality == 'CT') & ~pid_series.SeriesInstanceUID.isin(set_aside)]
    if s.empty:
        return None
    m, _ = main_series_rule(s.copy())
    return m.iloc[0] if len(m) else None

def prepare_with_set_aside(key, pid_series, uid, first_error):
    """After the rule's series (uid) gave no CT slices: set it aside, let the rule pick again, prepare that series with
    notebook 08's prepare_patient; at most MAX_SERIES_SET_ASIDE series are set aside."""
    sizes = pid_series.drop_duplicates('SeriesInstanceUID').set_index('SeriesInstanceUID').FileSize.astype(float)
    set_aside, res = [uid], {'key': key, 'status': 'failed', 'error': first_error}
    while len(set_aside) <= MAX_SERIES_SET_ASIDE:
        b = rule_main_without(pid_series, set_aside)
        if b is None:
            return {'key': key, 'status': 'failed', 'error': f'no other eligible CT series ({first_error})'}, set_aside, None
        res = prepare_patient(key, b.SeriesInstanceUID, sizes[b.SeriesInstanceUID])
        if res['status'] != 'failed':
            return res, set_aside, b.SeriesInstanceUID
        if NO_SLICES not in res['error']:
            return res, set_aside, None
        set_aside.append(b.SeriesInstanceUID)
    return res, set_aside, None

def use_series(key, uid, set_aside):
    """Point a new patient's rows (PATIENTS and its target's table) to the series actually used."""
    r = PATIENTS.loc[PATIENTS.key == key].iloc[0]
    s = SERIES_NEW[r.collection]
    b = s[s.SeriesInstanceUID == uid].iloc[0]
    for frame in (PATIENTS, TARGETS[r.target]):
        i = frame.index[frame.key == key]
        for c in SERIES_COLS:
            if c in frame and c in b:
                frame.loc[i, c] = b[c]
        frame.loc[i, 'series_set_aside'] = ';'.join(set_aside)

ASIDE_FILE = CACHE / 'series_set_aside.csv'
ASIDE_COLS = ['key', 'target', 'rule_series', 'rule_series_description', 'set_aside', 'used_series',
              'used_series_description', 'used_image_count', 'status', 'error', 'session_utc']
for frame in (PATIENTS, *TARGETS.values()):
    frame['series_set_aside'] = ''

def fake_part1():
    """Local code tests only: random headers and embeddings instead of images (seeded, label-free)."""
    rng = np.random.default_rng(7)
    for key in NEW.key:
        if not header_ok(key):
            write_json_atomic(header_path(key), {
                'key': key, 'prep_key': PREP_KEY, 'fake': True,
                'hdr_manufacturer': str(rng.choice(['GE MEDICAL SYSTEMS', 'SIEMENS', 'PHILIPS', 'TOSHIBA'])),
                'hdr_z_spacing': float(rng.choice([0.625, 1.25, 2.5, 5.0])), 'hdr_kvp': float(rng.choice([100, 120, 140])),
                'hdr_slice_thickness': float(rng.choice([1.25, 2.5, 5.0])), 'hdr_pixel_spacing': float(rng.uniform(.6, .9)),
                'hdr_recon_diameter': float(rng.uniform(300, 450)), 'hdr_tube_current': float(rng.uniform(100, 400)),
                'hdr_exposure': np.nan, 'hdr_ctdivol': np.nan, 'hdr_pitch': float(rng.choice([0.984, 1.375])),
                'hdr_kernel': str(rng.choice(['B30F', 'STANDARD', 'B', 'LUNG'])), 'hdr_filter_type': np.nan,
                'hdr_patient_position': 'FFS', 'hdr_contrast_agent_given': str(rng.choice(['yes', 'no'])),
                'body_area_cm2': float(rng.uniform(400, 900))})
        if not emb_ok(emb_path(key), VARIANTS):
            arrays = {f'ctfm__{v}': rng.normal(size=512).astype(np.float32) for v in VARIANTS}
            arrays.update({f'merlin__{v}': np.abs(rng.normal(size=2048)).astype(np.float32) for v in VARIANTS})
            np.savez(emb_path(key), meta=json.dumps({'emb_key': EMB_KEY, 'fake': True}), **arrays)

for k_old, folder, key_wanted in ((sorted(IN05_KEYS)[0], CACHE05, EMB_KEY05), (sorted(IN07_KEYS)[0], CACHE07, EMB_KEY),
                                  (sorted(IN08_KEYS)[0], CACHE08, EMB_KEY)):
    with np.load(folder / 'embeddings' / f'{k_old}.npz') as z:
        assert json.loads(str(z['meta']))['emb_key'] == key_wanted, f'{folder} embeddings were made with other settings'

t_part1 = time.time()
prep_failed, emb_failed = [], []
if TEST.get('fake_images'):
    fake_part1()
    MODELS = None
else:
    todo = [r for r in NEW.itertuples() if not emb_ok(emb_path(r.key), VARIANTS) or not header_ok(r.key)]
    print(f'New scans to prepare: {len(todo)} of {len(NEW)} (the others are finished in the cache)')
    if todo:
        res = Parallel(n_jobs=PREP_JOBS, return_as='generator_unordered')(
            delayed(prepare_patient)(r.key, r.SeriesInstanceUID, r.FileSize) for r in todo)
        for i, r in enumerate(res, 1):
            if r['status'] == 'failed':
                prep_failed.append(r)
                print(f"  FAILED {r['key']}: {r['error']}")
            if i % 25 == 0 or i == len(todo):
                print(f'  prepared {i}/{len(todo)} ({(time.time() - t_part1) / 60:.1f} min, '
                      f'{shutil.disk_usage(WORK).free / 1e9:.0f} GB free)')
    # amendment of 4 Nov 2026: series without CT slices are set aside and the rule picks again
    aside_rows = pd.read_csv(ASIDE_FILE).to_dict('records') if ASIDE_FILE.exists() else []
    no_ct = [f for f in prep_failed if NO_SLICES in f['error']]
    if no_ct:
        print(f'Series without CT slices: {len(no_ct)}; setting them aside and applying the rule again')
        jobs = []
        for f in no_ct:
            r = PATIENTS.loc[PATIENTS.key == f['key']].iloc[0]
            s = SERIES_NEW[r.collection]
            pid_series = s[s.PatientID == r.PatientID]
            first = rule_main_without(pid_series, [])
            assert first is not None and first.SeriesInstanceUID == r.SeriesInstanceUID, \
                f"{r.key}: the rule on this patient's series does not give its main series again"
            jobs.append((r, pid_series, f['error']))
        res = Parallel(n_jobs=PREP_JOBS)(delayed(prepare_with_set_aside)(r.key, ps, r.SeriesInstanceUID, e)
                                         for r, ps, e in jobs)
        now = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
        prep_failed = [f for f in prep_failed if NO_SLICES not in f['error']]
        for (r, ps, _), (out, set_aside, used) in zip(jobs, res):
            u = ps[ps.SeriesInstanceUID == used].iloc[0] if used else None
            aside_rows.append({'key': r.key, 'target': r.target, 'rule_series': r.SeriesInstanceUID,
                               'rule_series_description': r.SeriesDescription, 'set_aside': ';'.join(set_aside),
                               'used_series': used, 'used_series_description': u.SeriesDescription if used else None,
                               'used_image_count': u.ImageCount if used else None, 'status': out['status'],
                               'error': out.get('error'), 'session_utc': now})
            print(f"  {r.key}: set aside {len(set_aside)} -> "
                  f"{('used ' + str(u.SeriesDescription) + ' (' + str(u.ImageCount) + ' images)') if used else 'FAILED: ' + out['error']}")
            if out['status'] == 'failed':
                prep_failed.append(out)
    if aside_rows:
        ASIDE = pd.DataFrame(aside_rows, columns=ASIDE_COLS).drop_duplicates('key', keep='last')
        ASIDE.to_csv(ASIDE_FILE, index=False)
    MODELS = {'ctfm': load_ctfm().to(DEVICE), 'merlin': load_merlin().to(DEVICE)}
    todo = [k for k in NEW.key if not emb_ok(emb_path(k), VARIANTS)
            and all(p.exists() for p in nifti_paths(k).values())]
    print(f'New scans to embed: {len(todo)}')
    items = [(k, nifti_paths(k), 'original') for k in todo]   # background and silhouette use the original's crop box
    emb_failed = embed_items(items, emb_path, MODELS, label='original + background + silhouette')
    for f in emb_failed:
        print(f"  EMBEDDING FAILED {f['key']}: {f['error']}")
    for k in NEW.key:                    # embedded scans are not needed again: free the work disk
        if emb_ok(emb_path(k), VARIANTS):
            for p in nifti_paths(k).values():
                p.unlink(missing_ok=True)
print(f'Part 1 finished in {(time.time() - t_part1) / 60:.1f} min')

# the series actually used (amendment of 4 Nov 2026), also for patients prepared in an earlier session
ASIDE = pd.read_csv(ASIDE_FILE) if ASIDE_FILE.exists() else pd.DataFrame(columns=ASIDE_COLS)
for a in ASIDE[ASIDE.used_series.notna()].itertuples():
    use_series(a.key, a.used_series, a.set_aside.split(';'))
for r in PATIENTS.itertuples():
    used = header_of(r.key).get('SeriesInstanceUID')
    assert used is None or used == r.SeriesInstanceUID, f'{r.key}: the prepared series is not the one recorded'
if len(ASIDE):
    print(f'Series set aside (no CT slices): {len(ASIDE)} patients; replaced: {int(ASIDE.used_series.notna().sum())}')
    show(ASIDE[['key', 'rule_series_description', 'used_series_description', 'used_image_count', 'status']], 30)

source_keys = set(labels[labels.task_id == SOURCE].collection + '__' + labels[labels.task_id == SOURCE].PatientID)
DATASETS = {SOURCE: sorted(source_keys), **{t: d.key.tolist() for t, d in TARGETS.items()}}
complete = {k for keys in DATASETS.values() for k in keys if all(has_variant(k, v) for v in VARIANTS)}
inventory = PATIENTS[['key', 'target', 'collection', 'PatientID', 'SeriesInstanceUID', 'SeriesDescription', 'ImageCount',
                      'FileSize', 'series_set_aside']].merge(pd.DataFrame([header_of(k) for k in PATIENTS.key]).drop(
                          columns=['SeriesInstanceUID'], errors='ignore'), on='key', how='left')
inventory['embedded'] = inventory.key.isin(complete)
inventory.to_csv(CACHE / 'image_inventory.csv', index=False)
pd.DataFrame(prep_failed + emb_failed, columns=['key', 'status', 'error']).to_csv(CACHE / 'failures_this_session.csv',
                                                                                  index=False)
cov = pd.DataFrame([{'dataset': d, 'role': 'source' if d == SOURCE else ROLE[d], 'patients': len(keys),
                     'embedded': sum(k in complete for k in keys)} for d, keys in DATASETS.items()])
cov['coverage'] = cov.embedded / cov.patients
show(cov)
if (cov.coverage < GATE_MIN_COVERAGE).any():
    print(f'WARNING: fewer than {GATE_MIN_COVERAGE:.0%} of the patients of a dataset have embeddings. '
          'Run all again (failed downloads are retried); if it persists, inspect the failed-download log.')
if TEST.get('stop_after_images'):
    raise SystemExit('Local test: stopping after the image part.')

New scans to prepare: 17 of 515 (the others are finished in the cache)
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0187: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0246: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0230: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0210: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0194: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0227: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0232: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0214: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0205: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung_Dx-A0261: AssertionError('no image slices in the download')
  FAILED Lung-PET-CT-Dx__Lung

/tmp/ipykernel_615/4028487462.py:86: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  series = series.replace(r'^\s*$', np.nan, regex=True)
/tmp/ipykernel_615/4028487462.py:86: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  series = series.replace(r'^\s*$', np.nan, regex=True)
/tmp/ipykernel_615/4028487462.py:86: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd

  Lung-PET-CT-Dx__Lung_Dx-A0187: set aside 1 -> used Thorax  1.0  B31f (331 images)
  Lung-PET-CT-Dx__Lung_Dx-A0246: set aside 2 -> used Thorax  1.0  B31f (342 images)
  Lung-PET-CT-Dx__Lung_Dx-A0230: set aside 2 -> used Thorax  1.0  B70f (297 images)
  Lung-PET-CT-Dx__Lung_Dx-A0210: set aside 2 -> used Thorax  1.0  B70f (307 images)
  Lung-PET-CT-Dx__Lung_Dx-A0194: set aside 2 -> used Thorax  1.0  B31f (418 images)
  Lung-PET-CT-Dx__Lung_Dx-A0227: set aside 2 -> used Thorax  1.0  B31f (330 images)
  Lung-PET-CT-Dx__Lung_Dx-A0232: set aside 2 -> used Thorax  1.0  B31f (342 images)
  Lung-PET-CT-Dx__Lung_Dx-A0214: set aside 2 -> used CT WB  1.0  B30f (168 images)
  Lung-PET-CT-Dx__Lung_Dx-A0205: set aside 2 -> used Thorax  1.0  B31f (399 images)
  Lung-PET-CT-Dx__Lung_Dx-A0261: set aside 2 -> used Thorax  1.0  B31f (418 images)
  Lung-PET-CT-Dx__Lung_Dx-A0256: set aside 2 -> used Thorax  1.0  B31f (407 images)
  Lung-PET-CT-Dx__Lung_Dx-G0047: set aside 1 -> used Thorax  1.0  B70f (230 i

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  311MB            

... (2 outputs left out)

i3_resnet_clinical_longformer_best_clip_(…): downloading bytes:           |  0.00B            

New scans to embed: 17
  original + background + silhouette: 17/17 embedded (2.0 min)
Part 1 finished in 22.6 min
Series set aside (no CT slices): 17 patients; replaced: 17


,key,rule_series_description,used_series_description,used_image_count,status
0,Lung-PET-CT-Dx__Lung_Dx-A0187,<ALPHA Range>,Thorax 1.0 B31f,331,ok
1,Lung-PET-CT-Dx__Lung_Dx-A0246,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B31f,342,ok
2,Lung-PET-CT-Dx__Lung_Dx-A0230,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B70f,297,ok
3,Lung-PET-CT-Dx__Lung_Dx-A0210,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B70f,307,ok
4,Lung-PET-CT-Dx__Lung_Dx-A0194,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B31f,418,ok
5,Lung-PET-CT-Dx__Lung_Dx-A0227,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B31f,330,ok
6,Lung-PET-CT-Dx__Lung_Dx-A0232,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B31f,342,ok
7,Lung-PET-CT-Dx__Lung_Dx-A0214,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,CT WB 1.0 B30f,168,ok
8,Lung-PET-CT-Dx__Lung_Dx-A0205,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B31f,399,ok
9,Lung-PET-CT-Dx__Lung_Dx-A0261,Range-CT WB 1.0 B30f-Tra-<ALPHA Range>,Thorax 1.0 B31f,418,ok


,dataset,role,patients,embedded,coverage
0,lung histology (LUAD vs LUSC),source,104,104,1.0
1,NSCLC Radiogenomics,independent (notebook 08),199,199,1.0
2,CPTAC lung,one collection per class (notebook 08),73,73,1.0
3,NSCLC-Radiomics,"independent (new, plan 7j)",203,203,1.0
4,Lung-PET-CT-Dx,"independent (new, plan 7j)",312,312,1.0


## Part 2 · The four lung pairs (once on shuffled labels as a rehearsal, then on the real labels)

In [ ]:
# 10. Model building blocks (identical to notebooks 03 and 04), plus the same models on embeddings (notebook 08, unchanged)
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    return [list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y))
            for r in range(N_REPEATS)]

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, PA, PB):
    """Repeat-averaged AUC(A) - AUC(B) on the same patients, with a paired bootstrap 95% CI."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(np.mean([auc(y[idx], PA[r][idx]) - auc(y[idx], PB[r][idx]) for r in range(PA.shape[0])]))
    return mean_auc(y, PA) - mean_auc(y, PB), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_mean(x):
    rng = np.random.default_rng(SEED)
    x = np.asarray(x, float)
    vals = [x[rng.choice(len(x), len(x), replace=True)].mean() for _ in range(N_BOOT)]
    return float(x.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# --- metadata models (notebook 03) ---
def _fit_fold_df(df, y, cols, tr, te, k):
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(df.iloc[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[te])
    return te, proba

def cv_oof_df(df, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_df)(df, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- the same logistic regression on embeddings (standardized inside each training fold) ---
def _fit_fold_matrix(X, y, tr, te, k, extras):
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(X[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(X[te])
    out = {}
    for name, Xe in extras.items():           # the fitted fold model applied to other inputs of the same patients
        pe = np.full((len(te), k), np.nan)
        ok = np.where(~np.isnan(Xe[te]).any(axis=1))[0]
        if len(ok):
            tmp = np.zeros((len(ok), k))
            tmp[:, m.classes_] = m.predict_proba(Xe[te][ok])
            pe[ok] = tmp
        out[name] = pe
    return te, proba, out

def cv_oof_matrix(X, y, splits, extras=None):
    extras = extras or {}
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_matrix)(X, y, tr, te, k, extras) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    E = {n: np.full((len(splits), len(y), k), np.nan) for n in extras}
    for (r, _, _), (te, proba, out) in zip(jobs, res):
        P[r, te] = proba
        for n, pe in out.items():
            E[n][r, te] = pe
    return P, E

def _fit_fold_combined(df, E, y, cols, tr, te, k):
    ct = column_transformer(cols).fit(df.iloc[tr])
    sc = StandardScaler().fit(E[tr])
    Xtr = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[tr])), sparse.csr_matrix(sc.transform(E[tr]))]).tocsr()
    Xte = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[te])), sparse.csr_matrix(sc.transform(E[te]))]).tocsr()
    m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(Xte)
    return te, proba

def cv_oof_combined(df, E, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_combined)(df, E, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- exact permutation tests on one fixed label-independent split (notebook 03, 7b.6) ---
def encoded_folds_df(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def scaled_folds(X):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(X):
        sc = StandardScaler().fit(X[tr])
        out.append((tr, te, sc.transform(X[tr]), sc.transform(X[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(folds, y):
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

# --- within-group AUC and its exact test (notebook 04, part D) ---
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(folds, y, strata, keep):
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

In [ ]:
# 11. Datasets: notebook 05's task frame and embedding loader, notebook 07's input loader, the lung targets
def task_frame(tid, shuffle_seed=None):
    """Task patients with metadata, groups, headers and body size. shuffle_seed: shuffled labels (rehearsal)."""
    df = labels[labels.task_id == tid].merge(feat, on=KEY, how='left', validate='one_to_one')
    df['key'] = df.collection + '__' + df.PatientID
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if shuffle_seed is not None:
        y = np.random.default_rng(shuffle_seed).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(polish, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

def load_embeddings(keys, variants, path_of):
    out = {(e, v): [] for e in ENCODERS for v in variants}
    for k in keys:
        p = path_of(k)
        ok = p is not None and p.exists()
        z = np.load(p) if ok else None
        for e in ENCODERS:
            for v in variants:
                name = f'{e}__{v}'
                out[(e, v)].append(z[name] if ok and name in z.files else None)
        if z is not None:
            z.close()
    res = {}
    for (e, v), lst in out.items():
        d = next((len(a) for a in lst if a is not None), 1)
        res[(e, v)] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in lst]).astype(float)
    return res

def load_inputs(keys):
    """{(encoder, variant): matrix} from whichever notebook holds each scan's embedding."""
    out = {}
    for v in VARIANTS:
        out.update(load_embeddings(keys, (v,), lambda k, v=v: emb_file(k, v)))
    return out

def dataset_frame(name, shuffle_seed=None):
    """Patients with embeddings, labels in the shared class order (class 1: squamous). The source and NSCLC Radiogenomics
    use notebook 05's task frame and CPTAC lung notebook 08's frame, so their rows and shuffles are exactly those of
    notebooks 05, 07, 08 and 09; the new targets are built the same way (sorted by key)."""
    if name in SOURCES or name == 'NSCLC Radiogenomics':
        df, y, classes = task_frame(name if name in SOURCES else NSCLC_TASK, shuffle_seed)
        assert tuple(classes) == CLASSES[name], f'{name}: class order {classes} differs from {CLASSES[name]}'
    else:
        df = TARGETS[name]
        y = (df.label == CLASSES[name][1]).astype(int).to_numpy()
        if shuffle_seed is not None:
            y = np.random.default_rng(shuffle_seed).permutation(y)
    keep = df.key.isin(complete).to_numpy()
    return df[keep].reset_index(drop=True), y[keep], int((~keep).sum())

def with_headers(df):
    """The scan's header record and body size, prepared as in notebooks 05 and 06 (categorical fields as text)."""
    h = pd.DataFrame([header_of(k) for k in df.key])
    wanted = ['body_area_cm2', 'hdr_contrast_agent_given', 'hdr_coverage_mm', 'hdr_manufacturer'] + HDR_COLS
    df = df.drop(columns=[c for c in wanted if c in df]).merge(h[['key'] + [c for c in wanted if c in h]], on='key',
                                                                how='left')
    for c in wanted:
        if c not in df:
            df[c] = np.nan
    for c in HDR_CATEGORICAL + ['hdr_manufacturer']:
        df[c] = df[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
    df['manufacturer'] = df.hdr_manufacturer.str.upper().str.extract(r'^\s*([A-Z]+)', expand=False)   # as experiment C
    return df

In [ ]:
# 12. The analyses of plan 7j A (notebook 08's transfer, notebook 09's silhouette and field checks, the 7j reading)
def fit_apply(Xa, ya, Xb):
    """The probe (scaling + logistic regression, notebook 05's settings) fitted on all of A, applied to B. The score is
    the log-odds of class 1: the same ranking (and AUC) as the probability, but it cannot round to exactly 0 or 1 when
    B's embeddings lie far from A's (settled item 6)."""
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(Xa, ya)
    assert list(m.classes_) == [0, 1]
    return m.decision_function(Xb)

def as_P(s):
    """One score vector in the (repeats, patients, classes) shape of notebook 05's AUC and bootstrap functions
    (only the class-1 column is read, by rank)."""
    return np.stack([-s, s], axis=1)[None]

def _external_chunk(Xa, ys, Xb, yb):
    return [auc_binary(yb, fit_apply(Xa, yp, Xb)) for yp in ys]

def external_test(Xa, ya, Xb, yb):
    """External AUC; permutation test by refitting on permuted A labels (settled item 6: descriptive, and the gate)."""
    s = fit_apply(Xa, ya, Xb)
    observed = auc_binary(yb, s)
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(ya) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_external_chunk)(Xa, c, Xb, yb) for c in chunks))
    return s, observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), float(null.mean())

def boot_gap(ya, Pa, yb, sb):
    """Internal AUC of A (repeat-averaged out of fold) minus the AUC of the A-model on B; A and B resampled independently."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        ia = rng.choice(len(ya), len(ya), replace=True)
        ib = rng.choice(len(yb), len(yb), replace=True)
        if len(np.unique(ya[ia])) < 2 or len(np.unique(yb[ib])) < 2:
            continue
        vals.append(np.mean([auc(ya[ia], Pa[r][ia]) for r in range(Pa.shape[0])]) - auc_binary(yb[ib], sb[ib]))
    return mean_auc(ya, Pa) - auc_binary(yb, sb), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def reading(lo, hi, gap_lo, gap_hi):
    """Plan 7g, experiment B, with the edge cases of settled item 15 (both fixed before any data)."""
    if lo <= 0.5:
        return 'does not transfer'
    if gap_lo > 0:
        return 'transfers with a loss'
    return 'transfers'

def internal(X, y):
    """Notebook 05's probe inside one dataset: 5 x 5 CV, bootstrap CI, permutation test on a fixed split."""
    P, _ = cv_oof_matrix(X, y, repeated_splits(y))
    lo, hi = boot_ci(y, P)
    _, p, null = permutation_test(scaled_folds(X), y)
    return P, {'auc': mean_auc(y, P), 'ci_low': lo, 'ci_high': hi, 'perm_p': p, 'null_mean_auc': float(null.mean())}

def transfers(lo):
    """Notebook 08's rule (7g, settled item 15): a model transfers if its external CI lies wholly above 0.5."""
    return bool(lo > 0.5)

def boot_auc(y, x):
    """AUC of one field for class 1 (rank-based), with a percentile bootstrap CI over patients; missing values left out."""
    ok = np.isfinite(x)
    y, x = y[ok], x[ok]
    if len(np.unique(y)) < 2 or np.unique(x).size < 2:
        return np.nan, np.nan, np.nan, int(ok.sum())
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) == 2:
            vals.append(auc_binary(y[idx], x[idx]))
    return auc_binary(y, x), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)), int(ok.sum())

def recorded(path, key, col):
    """A number recorded by an earlier notebook for the same labels (local tests: other shuffles, so none)."""
    if TEST.get('real_is_shuffled') and '/real/' in path.as_posix():
        return np.nan
    t = pd.read_csv(path).set_index(list(key.keys()))
    return float(t.loc[tuple(key.values()), col])

def repro_row(quantity, value, ref):
    return {'quantity': quantity, 'value': value, 'recorded': ref,
            'reproduced': (bool(abs(value - ref) < REPRO_TOL) if np.isfinite(ref) else None)}

def reading_7j(transferred):
    """Plan 7j A.5, fixed before this notebook was written. transferred: {new target: did any background-only probe
    (CT-FM or Merlin) transfer by notebook 08's rule}."""
    n = sum(bool(v) for v in transferred.values())
    return ('consistent across independent cohorts' if n == 0 else
            'partly consistent' if n < len(transferred) else 'not confirmed')

def direction(lo, hi, ref_auc):
    """Plan 7j A.6: where a target's slice-thickness AUC (squamous higher) lies, relative to TCGA's side of 0.5."""
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(ref_auc)):
        return 'not available'
    if lo <= 0.5 <= hi:
        return 'no clear difference'
    side = 'lower' if hi < 0.5 else 'higher'
    return 'same as TCGA' if side == ('lower' if ref_auc < 0.5 else 'higher') else 'opposite to TCGA'

def analyse(mode, shuffle_seed):
    dfs, ys, ex_s = dataset_frame(SOURCE, shuffle_seed)
    dfs = with_headers(dfs)
    es = load_inputs(dfs.key)
    in05, in06 = IN05 / mode / 'image_probe.csv', IN06 / mode / 'silhouette_probe.csv'
    rec = {'transfer': [], 'silhouette': [], 'fields': [], 'repro': [], 'scores': []}
    src_int = {}
    for enc in ENCODERS:                 # the source's own AUCs, each checked against the notebook that computed it
        for v in VARIANTS:
            P, si = internal(es[(enc, v)], ys)
            src_int[(enc, v)] = (P, si)
            ref = recorded(in06, {'task_id': SOURCE, 'encoder': enc}, 'silhouette_auc') if v == 'silhouette' else \
                recorded(in05, {'task_id': SOURCE, 'encoder': enc}, f'{INPUT_NAME[v]}_auc')
            rec['repro'].append(repro_row(f'source internal AUC, {enc}, {INPUT_NAME[v]}', si['auc'], ref))
    frames = {SOURCE: (dfs, ys)}
    for tgt in OLD_TARGETS + NEW_TARGETS:
        pair = PAIR_OF[tgt]
        dft, yt, ex_t = dataset_frame(tgt, shuffle_seed)
        dft = with_headers(dft)
        frames[tgt] = (dft, yt)
        et = load_inputs(dft.key)
        base = {'pair': pair, 'source': SOURCE, 'target': tgt, 'role': ROLE[tgt], 'n_source': len(ys),
                'n_target': len(yt), 'n_target_excluded_no_image': ex_t,
                'source_classes': ' | '.join(f'{c}: {int((ys == j).sum())}' for j, c in enumerate(CLASSES[SOURCE])),
                'target_classes': ' | '.join(f'{c}: {int((yt == j).sum())}' for j, c in enumerate(CLASSES[tgt]))}
        for enc in ENCODERS:
            ext = {}
            for v in VARIANTS:
                Xs, Xt = es[(enc, v)], et[(enc, v)]
                assert not (np.isnan(Xs).any() or np.isnan(Xt).any()), 'missing embeddings'
                s_ext, a_ext, p_ext, null_ext = external_test(Xs, ys, Xt, yt)
                ext[v] = s_ext
                lo, hi = boot_ci(yt, as_P(s_ext))
                rec['scores'].append(pd.DataFrame({'pair': pair, 'encoder': enc, 'input': INPUT_NAME[v],
                                                   'key': dft.key, 'label': yt, 'score': s_ext}))
                if v in INPUTS_B:        # notebook 08's full analysis, with the target's own AUC and the reverse
                    Ps, si = src_int[(enc, v)]
                    Pt, ti = internal(Xt, yt)
                    gap, glo, ghi = boot_gap(ys, Ps, yt, s_ext)
                    s_rev, a_rev, p_rev, null_rev = external_test(Xt, yt, Xs, ys)
                    rlo, rhi = boot_ci(ys, as_P(s_rev))
                    row = {**base, 'encoder': enc, 'input': INPUT_NAME[v], 'source_internal_auc': si['auc'],
                           'source_internal_ci_low': si['ci_low'], 'source_internal_ci_high': si['ci_high'],
                           'target_internal_auc': ti['auc'], 'target_internal_ci_low': ti['ci_low'],
                           'target_internal_ci_high': ti['ci_high'], 'target_internal_perm_p': ti['perm_p'],
                           'target_internal_null_mean_auc': ti['null_mean_auc'],
                           'external_auc': a_ext, 'external_ci_low': lo, 'external_ci_high': hi,
                           'external_perm_p': p_ext, 'external_null_mean_auc': null_ext,
                           'external_reversed': bool(hi < 0.5), 'transfer_gap': gap, 'gap_ci_low': glo,
                           'gap_ci_high': ghi, 'reading': reading(lo, hi, glo, ghi),
                           'reverse_auc': a_rev, 'reverse_ci_low': rlo, 'reverse_ci_high': rhi,
                           'reverse_perm_p': p_rev, 'reverse_null_mean_auc': null_rev}
                    rec['transfer'].append(row)
                    if tgt in OLD_TARGETS:   # must reproduce notebook 08 (same rows, labels and functions)
                        t08 = IN08 / mode / 'transfer.csv'
                        k08 = {'pair': pair, 'encoder': enc, 'input': INPUT_NAME[v]}
                        for col in ('external_auc', 'external_ci_low', 'external_ci_high', 'target_internal_auc'):
                            rec['repro'].append(repro_row(f'{pair}, {enc}, {INPUT_NAME[v]}: {col}', row[col],
                                                          recorded(t08, k08, col)))
                else:                        # the silhouette (secondary): notebook 09's check 1
                    d, dlo, dhi = boot_diff(yt, as_P(ext['background']), as_P(s_ext))
                    srow = {**base, 'encoder': enc, 'source_internal_silhouette_auc': src_int[(enc, v)][1]['auc'],
                            'silhouette_external_auc': a_ext, 'silhouette_ci_low': lo, 'silhouette_ci_high': hi,
                            'silhouette_perm_p': p_ext, 'silhouette_null_mean_auc': null_ext,
                            'silhouette_transfers': transfers(lo), 'background_minus_silhouette': d,
                            'difference_ci_low': dlo, 'difference_ci_high': dhi}
                    rec['silhouette'].append(srow)
                    if tgt in OLD_TARGETS:   # must reproduce notebook 09
                        t09 = IN09 / mode / 'silhouette_transfer.csv'
                        rec['repro'].append(repro_row(f'{pair}, {enc}, silhouette external AUC', a_ext,
                                                      recorded(t09, {'pair': pair, 'encoder': enc},
                                                               'silhouette_external_auc')))
    # plan 7j A.6: which fields differ by class, in every lung dataset (class 1: squamous), descriptive
    for name, (df, y) in frames.items():
        for f in FIELDS_NUMERIC:
            a, lo, hi, n = boot_auc(y, df[f].to_numpy(float))
            rec['fields'].append({'dataset': name, 'field': f, 'n': n, 'auc_squamous_higher': a, 'ci_low': lo,
                                  'ci_high': hi, 'median_class0': float(np.nanmedian(df[f].to_numpy(float)[y == 0])),
                                  'median_class1': float(np.nanmedian(df[f].to_numpy(float)[y == 1])),
                                  'missing_share': float(df[f].isna().mean())})
            if name in (SOURCE,) + OLD_TARGETS:   # must reproduce notebook 09's check 3
                rec['repro'].append(repro_row(f'{name}: {f} AUC (squamous higher)', a,
                                              recorded(IN09 / mode / 'field_differences_numeric.csv',
                                                       {'dataset': name, 'field': f}, 'auc_squamous_higher')))
    return rec

In [ ]:
# 13. Run the analyses in one mode (rehearsal on shuffled labels, or real), with saving and a run record
def run_mode(mode):
    out = OUT_ROOT / mode
    out.mkdir(parents=True, exist_ok=True)
    shuffle_seed = 999 if mode == 'rehearsal' else (1000 if TEST.get('real_is_shuffled') else None)
    key = hashlib.md5(json.dumps({'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT,
                                  'N_PERM': N_PERM, 'LR': LR_PARAMS, 'EMB_KEY': EMB_KEY, 'EMB_KEY05': EMB_KEY05,
                                  'TEST': TEST, 'shuffle': shuffle_seed, 'complete': sorted(complete),
                                  'pairs': PAIRS, 'classes': CLASSES}, sort_keys=True, default=str).encode()).hexdigest()
    print(f'\n===== {mode.upper()} pass ({"shuffled labels" if shuffle_seed is not None else "REAL labels"}) =====')
    saved = out / 'analysis_cache.json'
    t0 = time.time()
    if saved.exists() and json.loads(saved.read_text()).get('settings_key') == key:
        rec = json.loads(saved.read_text())['rec']
        scores = pd.read_csv(out / 'transfer_scores.csv.gz')
        print('loaded from an earlier run')
    else:
        rec = analyse(mode, shuffle_seed)
        scores = pd.concat(rec.pop('scores'), ignore_index=True)
        scores.to_csv(out / 'transfer_scores.csv.gz', index=False)
        write_json_atomic(saved, {'settings_key': key, 'rec': rec})
    print(f'Analyses finished in {(time.time() - t0) / 60:.1f} min')

    transfer, sil = pd.DataFrame(rec['transfer']), pd.DataFrame(rec['silhouette'])
    fields, repro = pd.DataFrame(rec['fields']), pd.DataFrame(rec['repro'])
    # plan 7j A.5: the reading over the two new cohorts (background-only probes, both encoders)
    bg = transfer[transfer.input == 'background']
    moved = {t: bool((bg[bg.target == t].reading != 'does not transfer').any()) for t in NEW_TARGETS}
    verdict = {'background_transferred': moved, 'reading_7j': reading_7j(moved)}
    # plan 7j A.6: the slice-thickness prediction (descriptive)
    th = fields[fields.field == 'hdr_slice_thickness'].set_index('dataset')
    ref = float(th.auc_squamous_higher[SOURCE])
    pred = []
    for t in OLD_TARGETS + NEW_TARGETS:
        dirn = direction(th.ci_low[t], th.ci_high[t], ref)
        bt = bool((bg[bg.target == t].reading != 'does not transfer').any())
        pred.append({'target': t, 'role': ROLE[t], 'slice_thickness_auc': th.auc_squamous_higher[t],
                     'ci_low': th.ci_low[t], 'ci_high': th.ci_high[t], 'tcga_auc': ref, 'direction': dirn,
                     'background_transferred': bt,
                     'consistent_with_prediction': (not bt) or dirn == 'same as TCGA'})
    pred = pd.DataFrame(pred)
    tables = {'transfer.csv': transfer, 'silhouette_transfer.csv': sil, 'field_differences_numeric.csv': fields,
              'slice_thickness_prediction.csv': pred, 'reproduction_check.csv': repro}
    for name, t in tables.items():
        t.to_csv(out / name, index=False)
    (out / 'reading_7j.json').write_text(json.dumps(verdict, indent=2))
    show(transfer[['target', 'role', 'encoder', 'input', 'source_internal_auc', 'target_internal_auc', 'external_auc',
                   'external_ci_low', 'external_ci_high', 'reading']], 20)
    show(sil[['target', 'encoder', 'silhouette_external_auc', 'silhouette_ci_low', 'silhouette_ci_high',
              'silhouette_transfers']], 10)
    show(pred, 5)
    print('Plan 7j reading:', verdict)

    outputs = []
    for p in sorted(q for q in out.rglob('*') if q.is_file()):
        if p.name == 'run_manifest.json':
            continue
        n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
        outputs.append({'file': p.relative_to(out).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
    n11, md5_11 = cache_md5(CACHE)
    rep = repro.reproduced.dropna()
    manifest = {
        'settings_key': key, 'notebook': '11_external_lung.ipynb', 'mode': mode, 'plan': '7j (part A)',
        'labels': ('real' if shuffle_seed is None else 'shuffled (rehearsal)' if mode == 'rehearsal'
                   else 'shuffled (local code test)'), 'run_started_utc': RUN_STARTED,
        'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'in_colab': IN_COLAB, 'local_test_settings': TEST, 'input_runs': runs, 'input_caches': caches,
        'pairs': PAIRS, 'classes': CLASSES, 'expected_labels_new': EXPECTED_LABELS_NEW,
        'lung1_label_file': {'url': LUNG1_URL, 'md5': LUNG1_MD5}, 'dx_id_rule': DX_ID,
        'series_reply_md5': API_MD5, 'new_patients_without_main_series': NO_SERIES,
        'series_set_aside_amendment_2026_11_04': ASIDE.to_dict('records'),
        'main_series_rule_reproduces_nb01': RULE_REPRODUCED, 'same_versions_as_notebook05': same_versions,
        'reproduced_notebooks05_09': (bool(rep.astype(bool).all()) if len(rep) else None),
        'patients_new': int(len(PATIENTS)), 'coverage': cov.to_dict('records'), 'reading_7j': verdict,
        'settings': {**same, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM, 'SIL_HU': SIL_HU, 'REPRO_TOL': REPRO_TOL,
                     'PREP_KEY11': PREP_KEY, 'EMB_KEY11': EMB_KEY, 'INPUTS_B': INPUTS_B, 'VARIANTS': VARIANTS,
                     'MAX_SERIES_SET_ASIDE': MAX_SERIES_SET_ASIDE},
        'versions': VERSIONS, 'cache_files': {'count': n11, 'combined_md5': md5_11}, 'outputs': outputs,
    }
    (out / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return transfer, sil, repro

In [ ]:
# 14. Run: rehearsal (shuffled labels) -> automatic gate -> real labels
modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results:
        tr_r, sil_r, rep_r = results['rehearsal']
        nulls = pd.concat([tr_r[c] for c in ('target_internal_null_mean_auc', 'external_null_mean_auc',
                                             'reverse_null_mean_auc')] + [sil_r.silhouette_null_mean_auc])
        problems = []
        if len(tr_r) != len(PAIRS) * len(ENCODERS) * len(INPUTS_B) or len(sil_r) != len(PAIRS) * len(ENCODERS):
            problems.append('not every pair, encoder and input finished')
        if not nulls.between(*GATE_NULL_RANGE).all():
            problems.append(f'a null mean AUC is outside {GATE_NULL_RANGE}: {nulls.round(3).tolist()}')
        if (cov.coverage < GATE_MIN_COVERAGE).any():
            problems.append('too few scans embedded for a dataset')
        if problems:
            print('\nGATE NOT PASSED; the real-label pass was NOT run:', *problems, sep='\n - ')
            print('Inspect the run log.')
            break
        print('\nRehearsal gate passed: all analyses ran, null means look like chance, enough scans.')
        if not rep_r.reproduced.dropna().astype(bool).all():
            print('WARNING: the rehearsal did not reproduce notebooks 05-09 within the margin. The real pass runs, '
                  'but review the run log before interpreting it.')
    results[mode] = run_mode(mode)


===== REHEARSAL pass (shuffled labels) =====
Analyses finished in 21.0 min


,target,role,encoder,input,source_internal_auc,target_internal_auc,external_auc,external_ci_low,external_ci_high,reading
0,NSCLC Radiogenomics,independent (notebook 08),ctfm,image,0.436305,0.591128,0.503286,0.388098,0.616125,does not transfer
1,NSCLC Radiogenomics,independent (notebook 08),ctfm,background,0.413796,0.445309,0.448521,0.345618,0.558174,does not transfer
2,NSCLC Radiogenomics,independent (notebook 08),merlin,image,0.516660,0.500438,0.376050,0.277098,0.477174,does not transfer
3,NSCLC Radiogenomics,independent (notebook 08),merlin,background,0.549415,0.520518,0.460752,0.361109,0.568129,does not transfer
4,CPTAC lung,one collection per class (notebook 08),ctfm,image,0.436305,0.615766,0.542793,0.403896,0.672433,does not transfer
5,CPTAC lung,one collection per class (notebook 08),ctfm,background,0.413796,0.625075,0.419670,0.285598,0.556576,does not transfer
6,CPTAC lung,one collection per class (notebook 08),merlin,image,0.516660,0.501351,0.553303,0.420889,0.688137,does not transfer
7,CPTAC lung,one collection per class (notebook 08),merlin,background,0.549415,0.482282,0.440691,0.307538,0.581447,does not transfer
8,NSCLC-Radiomics,"independent (new, plan 7j)",ctfm,image,0.436305,0.530908,0.460397,0.369162,0.547107,does not transfer
9,NSCLC-Radiomics,"independent (new, plan 7j)",ctfm,background,0.413796,0.573452,0.422859,0.331885,0.510259,does not transfer


,target,encoder,silhouette_external_auc,silhouette_ci_low,silhouette_ci_high,silhouette_transfers
0,NSCLC Radiogenomics,ctfm,0.500365,0.392763,0.597587,False
1,NSCLC Radiogenomics,merlin,0.377327,0.278395,0.477706,False
2,CPTAC lung,ctfm,0.528529,0.393978,0.666704,False
3,CPTAC lung,merlin,0.490240,0.354884,0.628574,False
4,NSCLC-Radiomics,ctfm,0.435888,0.346286,0.522014,False
5,NSCLC-Radiomics,merlin,0.466976,0.380508,0.559320,False
6,Lung-PET-CT-Dx,ctfm,0.532558,0.447597,0.618053,False
7,Lung-PET-CT-Dx,merlin,0.485925,0.401817,0.568912,False


,target,role,slice_thickness_auc,ci_low,ci_high,tcga_auc,direction,background_transferred,consistent_with_prediction
0,NSCLC Radiogenomics,independent (notebook 08),0.428441,0.340875,0.520335,0.496168,no clear difference,False,True
1,CPTAC lung,one collection per class (notebook 08),0.419294,0.295446,0.552639,0.496168,no clear difference,False,True
2,NSCLC-Radiomics,"independent (new, plan 7j)",NaN,NaN,NaN,0.496168,not available,False,True
3,Lung-PET-CT-Dx,"independent (new, plan 7j)",0.503266,0.430935,0.569116,0.496168,no clear difference,False,True


Plan 7j reading: {'background_transferred': {'NSCLC-Radiomics': False, 'Lung-PET-CT-Dx': False}, 'reading_7j': 'consistent across independent cohorts'}

Rehearsal gate passed: all analyses ran, null means look like chance, enough scans.

===== REAL pass (REAL labels) =====
Analyses finished in 20.9 min


,target,role,encoder,input,source_internal_auc,target_internal_auc,external_auc,external_ci_low,external_ci_high,reading
0,NSCLC Radiogenomics,independent (notebook 08),ctfm,image,0.594917,0.608032,0.366009,0.267836,0.465425,does not transfer
1,NSCLC Radiogenomics,independent (notebook 08),ctfm,background,0.695845,0.620372,0.493428,0.391621,0.594600,does not transfer
2,NSCLC Radiogenomics,independent (notebook 08),merlin,image,0.763050,0.634502,0.480285,0.361132,0.597133,does not transfer
3,NSCLC Radiogenomics,independent (notebook 08),merlin,background,0.821380,0.570464,0.438846,0.337925,0.547419,does not transfer
4,CPTAC lung,one collection per class (notebook 08),ctfm,image,0.594917,0.454655,0.506757,0.369163,0.640564,does not transfer
5,CPTAC lung,one collection per class (notebook 08),ctfm,background,0.695845,0.448048,0.677177,0.552985,0.795629,transfers
6,CPTAC lung,one collection per class (notebook 08),merlin,image,0.763050,0.729880,0.695195,0.570116,0.809309,transfers
7,CPTAC lung,one collection per class (notebook 08),merlin,background,0.821380,0.603604,0.739489,0.609756,0.854571,transfers
8,NSCLC-Radiomics,"independent (new, plan 7j)",ctfm,image,0.594917,0.419866,0.542441,0.452651,0.635571,does not transfer
9,NSCLC-Radiomics,"independent (new, plan 7j)",ctfm,background,0.695845,0.451290,0.539087,0.441140,0.635891,does not transfer


,target,encoder,silhouette_external_auc,silhouette_ci_low,silhouette_ci_high,silhouette_transfers
0,NSCLC Radiogenomics,ctfm,0.458744,0.359279,0.560909,False
1,NSCLC Radiogenomics,merlin,0.438664,0.341721,0.533345,False
2,CPTAC lung,ctfm,0.637387,0.509803,0.760606,True
3,CPTAC lung,merlin,0.676426,0.536866,0.793364,True
4,NSCLC-Radiomics,ctfm,0.530702,0.433875,0.628109,False
5,NSCLC-Radiomics,merlin,0.595330,0.499990,0.690902,False
6,Lung-PET-CT-Dx,ctfm,0.479590,0.399939,0.560326,False
7,Lung-PET-CT-Dx,merlin,0.525113,0.444088,0.606341,False


,target,role,slice_thickness_auc,ci_low,ci_high,tcga_auc,direction,background_transferred,consistent_with_prediction
0,NSCLC Radiogenomics,independent (notebook 08),0.649142,0.558097,0.738723,0.337031,opposite to TCGA,False,True
1,CPTAC lung,one collection per class (notebook 08),0.359234,0.239086,0.487179,0.337031,same as TCGA,True,True
2,NSCLC-Radiomics,"independent (new, plan 7j)",NaN,NaN,NaN,0.337031,not available,False,True
3,Lung-PET-CT-Dx,"independent (new, plan 7j)",0.494350,0.421784,0.572358,0.337031,no clear difference,False,True


Plan 7j reading: {'background_transferred': {'NSCLC-Radiomics': False, 'Lung-PET-CT-Dx': False}, 'reading_7j': 'consistent across independent cohorts'}


In [ ]:
# 15. Plain-English summary, read back from the saved files
for mode in results:
    out = OUT_ROOT / mode
    tr = pd.read_csv(out / 'transfer.csv')
    sl = pd.read_csv(out / 'silhouette_transfer.csv')
    rp = pd.read_csv(out / 'reproduction_check.csv')
    lab_kind = json.loads((out / 'run_manifest.json').read_text())['labels']
    print(f"\n{mode.upper()} ({lab_kind}{': every number should look like chance' if lab_kind.startswith('shuffled') else ''})")
    for r in tr.itertuples():
        print(f"- {r.target} [{r.role}], {r.encoder}, {r.input}: source {r.source_internal_auc:.3f} -> external "
              f"{r.external_auc:.3f} ({r.external_ci_low:.3f}-{r.external_ci_high:.3f}); target's own "
              f"{r.target_internal_auc:.3f} -> {r.reading}")
    for r in sl.itertuples():
        print(f"- {r.target}, {r.encoder}, silhouette: external {r.silhouette_external_auc:.3f} "
              f"({r.silhouette_ci_low:.3f}-{r.silhouette_ci_high:.3f}) -> "
              f"{'transfers' if r.silhouette_transfers else 'does not transfer'}")
    print('- plan 7j reading:', json.loads((out / 'reading_7j.json').read_text())['reading_7j'])
    r_ok = rp.reproduced.dropna()
    print(f'- reproduced notebooks 05-09: {int(r_ok.astype(bool).sum())} of {len(r_ok)} checks')
print('\nNext: review the saved results and run log before writing up findings.')


REHEARSAL (shuffled (rehearsal): every number should look like chance)
- NSCLC Radiogenomics [independent (notebook 08)], ctfm, image: source 0.436 -> external 0.503 (0.388-0.616); target's own 0.591 -> does not transfer
- NSCLC Radiogenomics [independent (notebook 08)], ctfm, background: source 0.414 -> external 0.449 (0.346-0.558); target's own 0.445 -> does not transfer
- NSCLC Radiogenomics [independent (notebook 08)], merlin, image: source 0.517 -> external 0.376 (0.277-0.477); target's own 0.500 -> does not transfer
- NSCLC Radiogenomics [independent (notebook 08)], merlin, background: source 0.549 -> external 0.461 (0.361-0.568); target's own 0.521 -> does not transfer
- CPTAC lung [one collection per class (notebook 08)], ctfm, image: source 0.436 -> external 0.543 (0.404-0.672); target's own 0.616 -> does not transfer
- CPTAC lung [one collection per class (notebook 08)], ctfm, background: source 0.414 -> external 0.420 (0.286-0.557); target's own 0.625 -> does not transfer
-

In [ ]:
# 16. Stop the runtime once everything above has finished (notebook 08, unchanged)
# "Run all" stops at the first error, so if any cell fails, this cell never runs and the runtime stays on for checking.
if IN_COLAB:
    from google.colab import runtime
    print('Notebook finished:', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))
    drive.flush_and_unmount()   # push every result file to Drive before the machine is released
    print('Drive synced. Stopping the runtime now.')
    time.sleep(5)
    runtime.unassign()